In [1]:
import os
import sys
import torch
import numpy as np
import pyro
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

# Ajuste de ruta para ver 'src'
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

root_path = os.getcwd()
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data.loader import make_dataloaders
from src.models.lda import TopicModeler
from src.models.bnn import BayesianClassifier
from src.inference.variational import VariationalInference

# Limpiar el estado de Pyro por si acaso
pyro.clear_param_store()

/Users/emmarey/Library/Mobile Documents/com~apple~CloudDocs/Documents/ICAI/MÁSTER/SEGUNDO CUATRI/IA PROBABILISTCA/Bayesian-Sentiment-Analysis/ia_prob_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Cargar el modelo LDA que acabas de entrenar
lda_modeler = TopicModeler.load("experiments/results/lda_model.pkl")

# 2. Cargar TF-IDF y generar los vectores theta (Dirichlet) para todo el dataset
from src.data.loader import load_tfidf_splits
X_tr, X_val, X_te, _ = load_tfidf_splits()
theta_tr = lda_modeler.get_topics(X_tr)
theta_val = lda_modeler.get_topics(X_val)
theta_te = lda_modeler.get_topics(X_te)

# Concatenamos todos para que el loader sepa repartirlos
all_theta = np.vstack([theta_tr, theta_val, theta_te])

# 3. Crear DataLoaders (Aquí se combinan BERT + LDA)
train_loader, val_loader, test_loader = make_dataloaders(
    batch_size=64, 
    topic_vecs=all_theta
)

In [3]:
# Dimensiones: 768 (BERT) + 10 (LDA)
input_dim = 768 + 10
model = BayesianClassifier(input_dim=input_dim)
vi_engine = VariationalInference(model, lr=0.001)

print(f"Modelo inicializado con input_dim={input_dim}")

Modelo inicializado con input_dim=778


In [5]:
epochs = 50
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    for x_batch, y_batch in train_loader:
        # Paso de SVI para maximizar la ELBO 
        loss = vi_engine.train_step(x_batch, y_batch)
        total_loss += loss
    
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Evaluación en validación
    val_loss = vi_engine.evaluate_loss(val_loader)
    print(f"Epoch {epoch+1}/{epochs} | Train ELBO: {avg_loss:.4f} | Val ELBO: {val_loss:.4f}")

# Guardar los parámetros aprendidos de la posterior
os.makedirs("experiments/results", exist_ok=True)
pyro.get_param_store().save("experiments/results/bnn_params.pt")

Epoch 1/50 | Train ELBO: 558.9780 | Val ELBO: 525.4973
Epoch 2/50 | Train ELBO: 473.4317 | Val ELBO: 443.0273
Epoch 3/50 | Train ELBO: 396.2993 | Val ELBO: 370.1683
Epoch 4/50 | Train ELBO: 330.0682 | Val ELBO: 305.6666
Epoch 5/50 | Train ELBO: 272.4716 | Val ELBO: 252.9645
Epoch 6/50 | Train ELBO: 223.5962 | Val ELBO: 206.1938
Epoch 7/50 | Train ELBO: 182.5937 | Val ELBO: 166.6512
Epoch 8/50 | Train ELBO: 148.1924 | Val ELBO: 135.9322
Epoch 9/50 | Train ELBO: 120.3857 | Val ELBO: 110.6687
Epoch 10/50 | Train ELBO: 97.9652 | Val ELBO: 89.6786
Epoch 11/50 | Train ELBO: 79.9287 | Val ELBO: 74.3071
Epoch 12/50 | Train ELBO: 65.5585 | Val ELBO: 60.7709
Epoch 13/50 | Train ELBO: 54.4853 | Val ELBO: 50.2487
Epoch 14/50 | Train ELBO: 45.5624 | Val ELBO: 43.4905
Epoch 15/50 | Train ELBO: 38.3873 | Val ELBO: 36.5002
Epoch 16/50 | Train ELBO: 33.0489 | Val ELBO: 31.0906
Epoch 17/50 | Train ELBO: 28.8224 | Val ELBO: 27.5977
Epoch 18/50 | Train ELBO: 25.3711 | Val ELBO: 24.5844
Epoch 19/50 | Train